# M2 - Preparacion de datos
Objetivo: construir df_clean

In [1]:
import sys
import os
import pandas as pd

# Agregar la raiz del proyecto al path para importar
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Ruta al excel
DATA_PATH = os.path.join(ROOT, "data/raw", "2025 CPH.xlsx")

# Paso 1 - Carga y validacion inicial

In [2]:
from src.m2_data_prep.loader import cargar_hoja1

df_raw = cargar_hoja1(DATA_PATH, verbose = True)

[loader] Hoja1 cargada: 122 filas x 91 cols
[loader] Filas con FOLIO válido: 116
[loader] Validación OK: 116 filas, 91 columnas
[loader] Columnas: ['FECHA', 'FOLIO', 'DONADOR', 'RECEPTOR', 'NO. DE RECOLECCIÓN', 'SEXO DON', 'EDAD DON ', 'EDAD REC', 'TIPO', 'DX', 'ESTIMULACIÓN', 'DOSIS DE          G-CSF', 'DÍAS DE          G-CSF', 'DOSIS PLERIXAFOR mg/kg', 'PESO DON', 'TALLA DON', 'IMC DON', 'PESO REC.', 'CEBADO', 'TIEMPO MIN.', 'VOLEMIA mL', 'VOLEMIAS PROCESADAS', 'VOL. SANGUÍNEO PROCESADO mL', 'VOL. TOTAL PROCESADO mL', 'ACD TOT', 'VOL. PRODUCTO', 'VOLUMEN CALCULADO', 'DESECHABLE ', 'MAQUINA ', 'ACCESO', 'VEL. INICIAL mL/min', 'VEL. MEDIANA mL/min', 'VEL. FINAL mL/min', 'EVENTOS ADV EN PROCESO', 'GPO ABO DONADOR', ' RH DONADOR', 'CMV  IgM DONADOR', 'CMV IgG DONADOR', 'GPO ABO RECEPTOR', ' RH RECEPTOR', 'CMV  IgM RECEPTOR', 'CMV IgG RECEPTOR', 'PRE WBC  k/µL ', 'PRE MNC%', 'PRE MNC k/µL', 'PRE HTO %', 'PRE PLT k/μL', 'COSECHA WBC k/µL', 'COSECHA  MNC%', 'COSECHA MNC k/µL', ' COSECHA CMN

In [3]:
# Vista general
df_raw.head(3)

,FECHA,FOLIO,DONADOR,RECEPTOR,NO. DE RECOLECCIÓN,SEXO DON,EDAD DON,EDAD REC,TIPO,DX,...,EICH AGUDO,EICH CRONICO,SITIO AFECTADO,HCT-Cie,MORBILIDADES PRETRASPLANTE,COMPLICACIONES,INFECCIONES ESPECÍFICAS,MUERTE EN PRIMEROS 100 DÍAS,MUERTE EN PRIMER AÑO,MUERTE RELACIONADA AL TRASPLANTE
0,45671,25010026,NaN,NaN,1,M,56,56,AUTÓLOGO,MM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,45671,25010027,NaN,NaN,1,M,29,29,AUTÓLOGO,LH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,45672,25010030,NaN,NaN,1,M,66,66,AUTÓLOGO,MM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Shape y dtypes - todo debe de ser object
print("Shape:", df_raw.shape)
print("\nDtypes únicos:", df_raw.dtypes.value_counts().to_dict())

Shape: (116, 91)

Dtypes únicos: {<StringDtype(na_value=nan)>: 91}


In [5]:
# Lista completa de columnas con su índice
for i, col in enumerate(df_raw.columns):
    print(f"{i:>3}  {col}")

  0  FECHA
  1  FOLIO
  2  DONADOR
  3  RECEPTOR
  4  NO. DE RECOLECCIÓN
  5  SEXO DON
  6  EDAD DON 
  7  EDAD REC
  8  TIPO
  9  DX
 10  ESTIMULACIÓN
 11  DOSIS DE          G-CSF
 12  DÍAS DE          G-CSF
 13  DOSIS PLERIXAFOR mg/kg
 14  PESO DON
 15  TALLA DON
 16  IMC DON
 17  PESO REC.
 18  CEBADO
 19  TIEMPO MIN.
 20  VOLEMIA mL
 21  VOLEMIAS PROCESADAS
 22  VOL. SANGUÍNEO PROCESADO mL
 23  VOL. TOTAL PROCESADO mL
 24  ACD TOT
 25  VOL. PRODUCTO
 26  VOLUMEN CALCULADO
 27  DESECHABLE 
 28  MAQUINA 
 29  ACCESO
 30  VEL. INICIAL mL/min
 31  VEL. MEDIANA mL/min
 32  VEL. FINAL mL/min
 33  EVENTOS ADV EN PROCESO
 34  GPO ABO DONADOR
 35   RH DONADOR
 36  CMV  IgM DONADOR
 37  CMV IgG DONADOR
 38  GPO ABO RECEPTOR
 39   RH RECEPTOR
 40  CMV  IgM RECEPTOR
 41  CMV IgG RECEPTOR
 42  PRE WBC  k/µL 
 43  PRE MNC%
 44  PRE MNC k/µL
 45  PRE HTO %
 46  PRE PLT k/μL
 47  COSECHA WBC k/µL
 48  COSECHA  MNC%
 49  COSECHA MNC k/µL
 50   COSECHA CMN X10^9/ VOL TOTAL
 51  COSECHA HTO %
 52  CO

In [6]:
# Confirmar que el target existe
TARGET = "Processed WB (liters)"
assert TARGET in df_raw.columns, f"Columna target '{TARGET}' no encontrada"
print(f"Target '{TARGET}' presente en columna {df_raw.columns.tolist().index(TARGET)}")
print(df_raw[TARGET].head(10).tolist())

Target 'Processed WB (liters)' presente en columna 67
['25.568', '25.855', '20.907', '15.484', '9.993', '14.522', '15.722', '23.771', '13.144', '11.761']


# Paso 2 - Parseo numerico

In [7]:
from src.m2_data_prep.cleaning import parsear_numericos

df_parsed = parsear_numericos(df_raw, verbose = True)

[cleaning] Columnas numéricas generales : 57
[cleaning] Columnas CMV (serológicas)   : 4  → NEGATIVO=0, >250=250, número=float
[cleaning] Columnas RH  (binarias)      : 2  → POSITIVO=1, NEGATIVO=0
[cleaning] Columnas no-numéricas        : 28

[cleaning] Tokens inesperados (revisar):
  'REC MIELOIDE': 1 NaN extra - tokens: ['']

[cleaning] NaN por columna numérica:
  'EICH CRONICO'                               : 116 (100.0%)
  'POST CD34+/µL '                             : 116 (100.0%)
  'EICH AGUDO'                                 : 116 (100.0%)
  'REC PLAQ'                                   : 116 (100.0%)
  'REC MIELOIDE'                               : 116 (100.0%)
  'VIAB CRIO %'                                : 116 (100.0%)
  'CD34+/kg CRIO'                              : 116 (100.0%)
  'DOSIS PLERIXAFOR mg/kg'                     :  89 (76.7%)
  'CD34+ /µL DÍA 4'                            :  41 (35.3%)
  '% DE PÉRDIDA PLAQUETARIA'                   :   5 (4.3%)
  'CD34+/kg TRASP

In [8]:
# Verificar dtypes: deben aparecer float, int y str
print("Shape: ", df_parsed.shape)
print("Dtypes: ", df_parsed.dtypes.value_counts().to_dict())

Shape:  (116, 91)
Dtypes:  {dtype('float64'): 58, <StringDtype(na_value=nan)>: 28, dtype('int64'): 5}


In [9]:
# Inspeccion del target
print(df_parsed[TARGET].describe())

count    113.000000
mean      15.401186
std        5.821489
min        3.416000
25%       11.295000
50%       14.882000
75%       19.197000
max       28.908000
Name: Processed WB (liters), dtype: float64


In [10]:
from src.m2_data_prep.leakage import quitar_leakage

X, y = quitar_leakage(df_parsed, verbose = True)

[leakage] Columnas eliminadas : 58
  IDs / texto libre           : 13
  Leakage directo             : 9
  Leakage temporal            : 22
  Outcomes post-trasplante    : 13
  Target (y)                  : 1
[leakage] Columnas en X       : 33
[leakage] Test anti-leakage : sin correlaciones ≥ 0.99

[leakage] Features en X:
  'SEXO DON'
  'EDAD DON '
  'EDAD REC'
  'TIPO'
  'ESTIMULACIÓN'
  'DOSIS DE          G-CSF'
  'DÍAS DE          G-CSF'
  'DOSIS PLERIXAFOR mg/kg'
  'PESO DON'
  'TALLA DON'
  'IMC DON'
  'PESO REC.'
  'CEBADO'
  'VOLEMIA mL'
  'DESECHABLE '
  'MAQUINA '
  'ACCESO'
  'EVENTOS ADV EN PROCESO'
  'GPO ABO DONADOR'
  ' RH DONADOR'
  'CMV  IgM DONADOR'
  'CMV IgG DONADOR'
  'GPO ABO RECEPTOR'
  ' RH RECEPTOR'
  'CMV  IgM RECEPTOR'
  'CMV IgG RECEPTOR'
  'PRE WBC  k/µL '
  'PRE MNC%'
  'PRE MNC k/µL'
  'PRE HTO %'
  'PRE PLT k/μL'
  'CD34+ /µL DÍA 4'
  'CD34+ /µL DÍA 5'

[leakage] y = 'Processed WB (liters)'
  NaN en y: 3 / 116
  Rango: [3.416, 28.908] litros


In [11]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\ny describe:")
print(y.describe())

X shape: (116, 33)
y shape: (116,)

y describe:
count    113.000000
mean      15.401186
std        5.821489
min        3.416000
25%       11.295000
50%       14.882000
75%       19.197000
max       28.908000
Name: Processed WB (liters), dtype: float64


# Paso 4 - Encoding de categorias

In [12]:
from src.m2_data_prep.encoding import encodear_categorias

X_enc = encodear_categorias(X, verbose=True)

[encoding] Transformaciones aplicadas:
  SEXO DON: binaria: M=1, F=0
  TIPO: 3 cols (AUTÓLOGO, HAPLOIDÉNTICO, ALOGÉNICO)
  ESTIMULACIÓN: binaria (PEGILADO=1, resto=0)
  - CEBADO: binaria (SI=1, NO=0) — 2 NaN conservados
  DESECHABLE: binaria (IDL=1, ***=0)
  MAQUINA: 3 cols (EMT-25, EMT-50, EMT-68) — 2 NaN
  ACCESO: 3 cols (CATÉTER, PUNCIÓN, AMO)
  EVENTOS ADV EN PROCESO -> binaria (NINGUNO=0, cualquier evento=1)
  GPO ABO DONADOR -> one-hot 4 cols (A, AB, B, O)
  GPO ABO RECEPTOR -> one-hot 4 cols (A, AB, B, O) — A/O→NaN
  DOSIS PLERIXAFOR mg/kg → PLERIXAFOR_DADO (binaria: 1=administrado, 0=no administrado)

[encoding] Shape final de X: (116, 45)
[encoding] Sin columnas categóricas - X lista para PCA/SR


In [13]:
# Lista completa de features finales
for i, col in enumerate(X_enc.columns):
        print(f"{i:>2}  {col}")

 0  SEXO DON
 1  EDAD DON 
 2  EDAD REC
 3  DOSIS DE          G-CSF
 4  DÍAS DE          G-CSF
 5  PESO DON
 6  TALLA DON
 7  IMC DON
 8  PESO REC.
 9  CEBADO
10  VOLEMIA mL
11  DESECHABLE
12   RH DONADOR
13  CMV  IgM DONADOR
14  CMV IgG DONADOR
15   RH RECEPTOR
16  CMV  IgM RECEPTOR
17  CMV IgG RECEPTOR
18  PRE WBC  k/µL 
19  PRE MNC%
20  PRE MNC k/µL
21  PRE HTO %
22  PRE PLT k/μL
23  CD34+ /µL DÍA 4
24  CD34+ /µL DÍA 5
25  TIPO_ALOGÉNICO
26  TIPO_AUTÓLOGO
27  TIPO_HAPLOIDÉNTICO
28  ESTIMULACION_PEGILADO
29  MAQUINA_EMT-25
30  MAQUINA_EMT-50
31  MAQUINA_EMT-68
32  ACCESO_AMO
33  ACCESO_CATÉTER
34  ACCESO_PUNCIÓN
35  EVENTO_ADV
36  ABO_DON_A
37  ABO_DON_AB
38  ABO_DON_B
39  ABO_DON_O
40  ABO_REC_A
41  ABO_REC_AB
42  ABO_REC_B
43  ABO_REC_O
44  PLERIXAFOR_DADO


In [14]:
# Verificar NaN por columna en X_enc
nan_counts = X_enc.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print("NaN por columna:")
print(nan_counts)

NaN por columna:
CD34+ /µL DÍA 4      41
CD34+ /µL DÍA 5       4
VOLEMIA mL            3
PRE PLT k/μL          2
MAQUINA_EMT-68        2
MAQUINA_EMT-50        2
MAQUINA_EMT-25        2
CEBADO                2
PRE HTO %             2
PRE MNC k/µL          2
PRE MNC%              2
PRE WBC  k/µL         2
CMV IgG RECEPTOR      1
CMV  IgM RECEPTOR     1
ABO_REC_A             1
ABO_REC_AB            1
ABO_REC_B             1
ABO_REC_O             1
dtype: int64


# Paso 5 - Persistir df_clean
Combinamos X_enc + y en un unico DataFrame y lo guardamos como parquete

In [15]:
from pathlib import Path

# Combinar X_enc + y
df_clean = X_enc.copy()
df_clean["Processed WB (liters)"] = y.values

# Guardar
out_path = Path(ROOT) / "data" / "processed" / "df_clean.csv"
out_path.parent.mkdir(parents = True, exist_ok = True)
df_clean.to_csv(out_path, index = False)

print(f"df_clean guardado en: {out_path}")
print(f"Shape: {df_clean.shape}")
print(f"Tamaño: {out_path.stat().st_size / 1024:.1f} KB")

df_clean guardado en: /Users/robertosanchezsantoyo/Library/Mobile Documents/com~apple~CloudDocs/00_Main/03_Academic/Research/Investigacion_verano_2026/data/processed/df_clean.csv
Shape: (116, 46)
Tamaño: 24.3 KB


In [16]:
# Verificar leyendo el parquet desde cero
df_check = pd.read_csv(out_path)
print("Shape:", df_check.shape)
print("NaN en y:", df_check["Processed WB (liters)"].isna().sum())
df_check.head(3)

Shape: (116, 46)
NaN en y: 3


,SEXO DON,EDAD DON,EDAD REC,DOSIS DE G-CSF,DÍAS DE G-CSF,PESO DON,TALLA DON,IMC DON,PESO REC.,CEBADO,...,ABO_DON_A,ABO_DON_AB,ABO_DON_B,ABO_DON_O,ABO_REC_A,ABO_REC_AB,ABO_REC_B,ABO_REC_O,PLERIXAFOR_DADO,Processed WB (liters)
0,1,56,56,900,4,84.0,1.78,26.511804,84.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,25.568
1,1,29,29,900,4,90.0,1.74,29.726516,90.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,25.855
2,1,66,66,600,4,68.0,1.64,25.282570,68.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,20.907
